# Fisher Z transformations
### By Rohit Daniel

## 0. Import required packages

In [1]:
import camelot
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy.stats import norm
from itertools import product
from statsmodels.stats.meta_analysis import effectsize_smd

C:\Users\rodan\.conda\envs\jax\lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [2]:
from datetime import datetime
from IPython.display import Markdown, display

display(Markdown(f"**Notebook last run on:** {datetime.now():%Y-%m-%d %H:%M}"))

**Notebook last run on:** 2026-02-10 10:24

## 1. Overview

To enable synthesis across experimental and correlational study designs, all effects were expressed in terms of a common **associational effect metric**, the Fisher’s $z$ scale.

## 2. Define custom functions

In [3]:
def compute_r_to_fisher_z(df, alpha=0.05, clip_r=0.999999999):
    """
    Compute Fisher's z effect sizes for correlational studies (Pearson r).

    This function:
    - Treats Pearson's r as the primary effect-size statistic
    - Transforms r to Fisher's z for meta-analytic inference
    - Computes sampling variance, standard error, and confidence intervals

    Outputs Fisher_z, SE_z, VI_z, and confidence intervals
    on both the z and r scales.

    Required columns in df:
      - 'r' : Pearson correlation coefficient
      - 'N' : sample size used to compute r
    """

    # Critical value for N(0,1) at given alpha (two-sided)
    z_crit = scipy.stats.norm.ppf(1-alpha/2)

    results = []

    # Iterate row-wise to allow heterogeneous metadata across studies
    for _, row in df.iterrows():

        # --- Extract r and N -----------------------------------------
        r = row.get("r", np.nan)
        n = row.get("N", np.nan)

        try:
            r = float(r)
        except Exception:
            r = np.nan

        try:
            n = int(n)
        except Exception:
            n = np.nan

        # --- Extract metadata -------------------------------------
        variable_A = row.get("Variable_A", None)
        variable_B = row.get("Variable_B", None)
        feedback_type = row.get("Feedback_Type", None)
        attribution_type = row.get("Attribution_Type", None)
        setting = row.get("Setting", None)

        # --- Fisher's z computation ----------------------------------------
        # Fisher's z is defined only when:
        #   - sample size is greater than 3
        #   - r is finite and strictly between -1 and 1
        if isinstance(n, (int, np.integer)) and n > 3 and np.isfinite(r):

            # Clip r to avoid atanh(±1) → ±∞
            r_clip = float(np.clip(r, -clip_r, clip_r))

            # Fisher's z transformation
            fisher_z = np.arctanh(r_clip) 

            # Sampling variance of Fisher's z
            vi_z = 1.0 / (n - 3)

            # Standard error on the z scale
            se_z = np.sqrt(vi_z)

            # Confidence interval on the z scale
            ci_z_low = fisher_z - z_crit * se_z
            ci_z_high = fisher_z + z_crit * se_z

            # Back-transform CI limits to r for interpretability
            ci_r_low = np.tanh(ci_z_low)
            ci_r_high = np.tanh(ci_z_high)

        else:
            # If r or N are invalid, return NaNs for all inferential quantities
            fisher_z = np.nan
            se_z = np.nan
            vi_z = np.nan
            ci_z_low = np.nan
            ci_z_high = np.nan
            ci_r_low = np.nan
            ci_r_high = np.nan

        # --- Assemble output row -------------------------------------------
        row_out = {
            "Study": row.get("Study", None),
            "Study_Type": "Correlational",
            "Variable_A": variable_A,
            "Variable_B": variable_B,
            "Feedback_Type": feedback_type,
            "Attribution_Type": attribution_type,
            "Setting": setting,

            # Record the transformation used for traceability
            "Point_Transformation": "r_to_fisher_z",

            "N": n,

            # Original statistic
            "r": r,

            # Fisher's z outputs (used for meta-analysis)
            "Fisher_z": fisher_z,
            "SE_z": se_z,
            "VI_z": vi_z,

            # Confidence intervals on both scales
            "CI_lower_z": ci_z_low,
            "CI_upper_z": ci_z_high,
            "CI_lower_r": ci_r_low,
            "CI_upper_r": ci_r_high,
        }

        results.append(row_out)

    return pd.DataFrame(results)

In [4]:
def compute_cohen_d_to_fisher_z(
    df,
    control_label="Control",
    pre_post_r=0.75,  # Assumed pre-post correlation based on typical values reported in educational research (Papadopoulos, 2025)
    alpha=0.05,
    clip_r=0.999999999,
    include_control_setting=True,
):
    """
    Compute experimental effect sizes (Cohen's d / Hedges' g), convert them to an r-type effect,
    then transform to Fisher's z for synthesis on a common inferential scale.

    This function is designed for a pipeline where:
      1) Experimental studies are summarized via standardized mean differences (SMDs).
      2) SMDs are converted to effect-size correlations (r_effect).
      3) r_effect is transformed to Fisher's z (Fisher_z) with variance VI_z = 1/(N-3).
      4) Meta-analysis is performed on Fisher_z using inverse-variance weighting.

    REQUIRED COLUMNS
    ----------------
      ['Study','Predictor','Outcome','Setting','Condition','Design','Mean_diff','SD','N','Score_type']

    Design ∈ {"between_groups", "within_groups"}

    Mean_diff interpretation
    ------------------------
    - between_groups: group mean of the analyzed outcome (post-only *or* change-score mean),
                     depending on Score_type.
    - within_groups : mean change score (post − pre), paired within individuals.

    Notes
    -----
    - For between-groups designs, r_effect is derived from d via point-biserial equivalence:
        r = d / sqrt(d^2 + 4)
    - For within-groups (paired) designs, r_effect is derived via the t-equivalence:
        t = d * sqrt(n),  r = t / sqrt(t^2 + df)  with df = n - 1
      This yields an effect-size correlation representing association between time and outcome.
    - Fisher-z variance uses VI_z = 1/(N - 3). For within-groups, N is the sample size n.
    """
    # Critical value for N(0,1) at given alpha (two-sided)
    z_crit = scipy.stats.norm.ppf(1-alpha/2)

    # Clip bound protects against atanh(±1) -> ±inf due to rounding.
    if not (0 < clip_r < 1):
        raise ValueError("clip_r must be in (0, 1).")

    # --- Helper functions --------------------------------------------------

    def hedges_J(df_):
        """
        Hedges' small-sample correction factor J for Cohen's d.
        df_ is:
          - n_T + n_C - 2 for between-groups
          - n - 1 for within-groups (paired)
        """
        return 1 - (3 / (4 * df_ - 1)) if df_ > 1 else 1

    def d_to_r_between(d):
        """Convert between-groups Cohen's d to an equivalent point-biserial correlation."""
        return d / np.sqrt(d**2 + 4)

    def t_to_r(t, df_):
        """Convert t statistic to correlation: r = t / sqrt(t^2 + df)."""
        return t / np.sqrt(t**2 + df_)

    def fisher_z_and_ci(r, N):
        """
        Transform r to Fisher's z and compute:
          - Fisher_z
          - SE_z
          - VI_z
          - CI on z scale
          - CI back-transformed to r

        Uses:
          VI_z = 1/(N-3), SE_z = sqrt(VI_z)
          z_CI = z ± z_crit * SE_z
          r_CI = tanh(z_CI)

        Returns a 7-tuple:
          (z, se, vi, z_lo, z_hi, r_lo, r_hi)
        """
        # Fisher-z inference requires N > 3 and finite r
        if (not np.isfinite(r)) or (N is None) or (N <= 3):
            return (np.nan,) * 7

        # Clip r to avoid infinite z
        r = float(np.clip(r, -clip_r, clip_r))

        z = np.arctanh(r)
        vi = 1.0 / (N - 3)
        se = np.sqrt(vi)

        z_lo = z - z_crit * se
        z_hi = z + z_crit * se

        r_lo = np.tanh(z_lo)
        r_hi = np.tanh(z_hi)

        return z, se, vi, z_lo, z_hi, r_lo, r_hi

    # Collect results row-by-row (one output per contrast or condition)
    results = []

    # Group by (Study, Outcome) so each block uses exactly one design type.
    for (study, outcome), df_s in df.groupby(["Study", "Outcome"], dropna=False):

        # Enforce exactly one design per (Study, Outcome) block.
        designs = df_s["Design"].dropna().unique()
        if len(designs) != 1:
            raise ValueError(
                f"Mixed designs within Study='{study}', Outcome='{outcome}': {designs}"
            )
        design = designs[0]

        # Metadata assumed stable within (Study, Outcome)
        predictor = df_s["Predictor"].iloc[0] if "Predictor" in df_s.columns else None
        feedback_type = df_s["Feedback_Type"].iloc[0] if "Feedback_Type" in df_s.columns else None
        attribution_type = df_s["Attribution_Type"].iloc[0] if "Attribution_Type" in df_s.columns else None
        score_type_default = df_s["Score_type"].iloc[0] if "Score_type" in df_s.columns else None

        # ===================== BETWEEN-GROUPS ===============================
        if design == "between_groups":
            # Partition into control vs all non-control conditions (each treatment contrasted to control)
            control = df_s[df_s["Condition"] == control_label]
            treat = df_s[df_s["Condition"] != control_label]

            # Must have one control row and at least one treatment row
            if control.empty or treat.empty:
                raise ValueError(
                    f"Between-groups requires control ('{control_label}') and >=1 treatment "
                    f"for Study='{study}', Outcome='{outcome}'."
                )

            # If there are multiple control rows for the same outcome, the contrast becomes ambiguous
            if len(control) > 1:
                raise ValueError(
                    f"Multiple control rows found for Study='{study}', Outcome='{outcome}'. "
                    f"Ensure one '{control_label}' row per outcome."
                )

            # Extract control group stats once (shared across contrasts)
            n_c = int(control["N"].iloc[0])
            m_c = float(control["Mean_diff"].iloc[0])
            sd_c = float(control["SD"].iloc[0])
            setting_c = control["Setting"].iloc[0] if "Setting" in control.columns else None

            # Compute a separate contrast for each treatment condition vs control
            for _, row in treat.iterrows():
                n_t = int(row["N"])
                m_t = float(row["Mean_diff"])
                sd_t = float(row["SD"])
                setting_t = row.get("Setting", None)  # per-treatment setting

                # Total N for this contrast (used for Fisher-z variance in this implementation)
                N = n_c + n_t
                df_t = N - 2  # degrees of freedom for pooled SD and Hedges correction

                # Guard against degenerate inputs (e.g., missing/zero SD or tiny N)
                if min(n_c, n_t) <= 1 or sd_c <= 0 or sd_t <= 0:
                    d = g = r_eff = np.nan
                    z = se_z = vi_z = np.nan
                    ci_z_lo = ci_z_hi = ci_r_lo = ci_r_hi = np.nan
                else:
                    # Pooled SD for independent groups
                    sd_pooled = np.sqrt(((n_c - 1) * sd_c**2 + (n_t - 1) * sd_t**2) / df_t)

                    # Cohen's d: standardized mean difference (treatment vs control)
                    d = (m_t - m_c) / sd_pooled

                    # Hedges' g (bias-corrected d); retained for descriptive reporting
                    J = hedges_J(df_t)
                    g = d * J

                    # Convert d to an r-type effect size (point-biserial equivalence)
                    r_eff = d_to_r_between(d)

                    # Convert r to Fisher's z + sampling variance + CI
                    z, se_z, vi_z, ci_z_lo, ci_z_hi, ci_r_lo, ci_r_hi = fisher_z_and_ci(r_eff, N)

                # Assemble output record (one per treatment contrast)
                out = {
                    "Study": study,
                    "Study_Type": "Experimental",
                    "Predictor": predictor,
                    "Outcome": outcome,
                    "Feedback_Type": feedback_type,
                    "Attribution_Type": attribution_type,

                    # Setting corresponds to the *treatment* condition row (per your requirement)
                    "Setting": setting_t,
                    "Condition": row["Condition"],

                    "Design": design,
                    "Score_type": row.get("Score_type", score_type_default),
                    "Point_Transformation": "cohen_d_to_fisher_z",
                    "N": N,

                    # Intermediate SMD metrics (often useful for sanity checks / reporting)
                    "Cohens_d": d,
                    "Hedges_g": g,

                    # Harmonized associational metric for synthesis
                    "r_effect": r_eff,
                    "Fisher_z": z,
                    "SE_z": se_z,
                    "VI_z": vi_z,
                    "CI_lower_z": ci_z_lo,
                    "CI_upper_z": ci_z_hi,
                    "CI_lower_r": ci_r_lo,
                    "CI_upper_r": ci_r_hi,
                }

                # Optional: retain control setting for transparency / auditing
                if include_control_setting:
                    out["Control_Setting"] = setting_c

                results.append(out)

        # ====================== WITHIN-GROUPS ===============================
        elif design == "within_groups":
            # Each row is one within-group pre–post estimate for a given condition/outcome
            for _, row in df_s.iterrows():
                n = int(row["N"])
                mean_diff = float(row["Mean_diff"])
                sd_change = float(row["SD"])
                setting_row = row.get("Setting", None)  # per-condition setting

                # Guard against invalid inputs
                if n <= 1 or (not np.isfinite(sd_change)) or sd_change <= 0:
                    d = g = r_eff = np.nan
                    z = se_z = vi_z = np.nan
                    ci_z_lo = ci_z_hi = ci_r_lo = ci_r_hi = np.nan
                else:
                    # Standardized mean change using SD of change (often denoted d_z)
                    d = mean_diff / sd_change

                    # Hedges' correction (descriptive)
                    df_t = n - 1
                    J = hedges_J(df_t)
                    g = d * J

                    # Convert d_z to paired t, then to r:
                    # t = d * sqrt(n), df = n - 1, r = t / sqrt(t^2 + df)
                    t = d * np.sqrt(n)
                    r_eff = t_to_r(t, df_t)

                    # Fisher z + variance + CI
                    z, se_z, vi_z, ci_z_lo, ci_z_hi, ci_r_lo, ci_r_hi = fisher_z_and_ci(r_eff, n)

                results.append({
                    "Study": study,
                    "Study_Type": "Experimental",
                    "Predictor": predictor,
                    "Outcome": outcome,
                    "Feedback_Type": feedback_type,
                    "Attribution_Type": attribution_type,

                    # Per-condition setting
                    "Setting": setting_row,
                    "Condition": row["Condition"],

                    "Design": design,
                    "Score_type": row.get("Score_type", score_type_default),
                    "Point_Transformation": "cohen_d_to_fisher_z",
                    "N": n,

                    "Cohens_d": d,
                    "Hedges_g": g,

                    "r_effect": r_eff,
                    "Fisher_z": z,
                    "SE_z": se_z,
                    "VI_z": vi_z,
                    "CI_lower_z": ci_z_lo,
                    "CI_upper_z": ci_z_hi,
                    "CI_lower_r": ci_r_lo,
                    "CI_upper_r": ci_r_hi,
                })

        else:
            raise ValueError(f"Unknown Design value: {design}")

    return pd.DataFrame(results)

## 2. Correlational Studies

Associations between the key constructs—students’ mathematics self-efficacy, students’ causal attributions, and teacher feedback—were extracted from each eligible study. Reported Pearson correlation coefficients ($r$), along with corresponding sample sizes ($n$), were used as the primary effect-size information. When reported, $p$ values were retained for descriptive purposes but were not used in effect-size computation or weighting.

Statistical inference for correlational effects was conducted on the Fisher’s $z$ scale, which provides a variance-stabilizing transformation for Pearson’s $r$ and yields an approximately normal sampling distribution. Each correlation coefficient was transformed as:

$$
z = \operatorname{atanh}(r) = \frac{1}{2}\ln\left(\frac{1+r}{1-r}\right).
$$

For studies with sample size $n > 3$, the sampling variance of Fisher’s $z$ is given by:

$$
v_z = \frac{1}{n - 3},
$$

with corresponding standard error:

$$
SE_z = \frac{1}{\sqrt{n - 3}}.
$$

A 95\% confidence interval (CI) was constructed on the Fisher’s $z$ scale as:

$$
z_{L,U} = z \pm 1.96 \cdot SE_z
$$

### CS 1: The Development of English and Mathematics Self-Efficacy: A Latent Growth Curve Analysis (Phan, 2012)

In [5]:
# Extracted from Figure 3 on page 204
cs1 = pd.DataFrame({
    'Study': ['Phan (2012)'],
    'Variable_A': ['Feedback_Verbal_Persuation'],
    'Variable_B': ['Self-Efficacy'],
    'Feedback_Type': ['Formative'],
    'Attribution_Type': [None],
    'Setting': ['Classroom'],
    'r': [0.25],
    'N': [339],
    'p_value': ['p < .001']
})

In [6]:
print(cs1)

         Study                  Variable_A     Variable_B Feedback_Type  \
0  Phan (2012)  Feedback_Verbal_Persuation  Self-Efficacy     Formative   

  Attribution_Type    Setting     r    N   p_value  
0             None  Classroom  0.25  339  p < .001  


### CS 2: How growth mindset influences mathematics achievements: A study of Chinese middle school students (Dong, 2023)

In [7]:
# Extracted from Table 2 on page 6
cs2 = pd.DataFrame({
    'Study': ['Dong (2023)'],
    'Variable_A': ['Attributions_Failure_Internal'],
    'Variable_B': ['Self-Efficacy'],
    'Feedback_Type': [None],
    'Attribution_Type': ['Adaptive'],
    'Setting': ['Classroom'],
    'r': [0.52],
    'N': [266],
    'p_value': ['p < .01']
})

In [8]:
print(cs2)

         Study                     Variable_A     Variable_B Feedback_Type  \
0  Dong (2023)  Attributions_Failure_Internal  Self-Efficacy          None   

  Attribution_Type    Setting     r    N  p_value  
0         Adaptive  Classroom  0.52  266  p < .01  


### CS 3: Comparison of teacher and student perceptions of formative assessment feedback practices and association with individual student characteristics (van der Kleij, 2019)

In [9]:
# Extracted from Table 9 on page 185
cs3 = pd.DataFrame({
    'Study': ['van der Kleij (2019)'],
    'Variable_A': ['Self-Efficacy'], 
    'Variable_B': ['Feedback_Quality_Perceptions'],
    'Feedback_Type': ['Formative'],
    'Attribution_Type': [None],
    'Setting': ['Classroom'],
    'r': [0.32],
    'N': [174],
    'p_value': ['p < .001']
})

In [10]:
print(cs3)

                  Study     Variable_A                    Variable_B  \
0  van der Kleij (2019)  Self-Efficacy  Feedback_Quality_Perceptions   

  Feedback_Type Attribution_Type    Setting     r    N   p_value  
0     Formative             None  Classroom  0.32  174  p < .001  


### CS 4: Examining Achievement Goals and Causal Attributions Together as Predictors of Academic Functioning (Wolters et al., 2013)

In [11]:
# Extracted from Table 1 on page 306
cs4 = pd.DataFrame({
    'Study': ['Wolters et al. (2013)'] * 7,
    'Variable_A': ['Self-Efficacy'] * 7,
    'Variable_B': [
        'Attributions_Success_Ability',
        'Attributions_Success_Effort',
        'Attributions_Success_Environment',
        'Attributions_Failure_Ability',
        'Attributions_Failure_Effort',
        'Attributions_Failure_Environment',
        'Attributions_Failure_Task',
    ],
    'Feedback_Type': [None] * 7,
    'Attribution_Type': ['Adaptive', 'Adaptive', 'Maladaptive', 'Maladaptive', 'Adaptive', 'Maladaptive', 'Maladaptive'],
    'Setting': ['Classroom'] * 7,   
    'r': [ 0.20, -0.72, -0.16, -0.33, -0.08, -0.01, 0.71],
    'N': [224] * 7,
    'p_value': ['p < .01', 'p < .001', 'p < .05', 'p < .001', 'ns', 'ns', 'p < .001'],
})

In [12]:
print(cs4)

                   Study     Variable_A                        Variable_B  \
0  Wolters et al. (2013)  Self-Efficacy      Attributions_Success_Ability   
1  Wolters et al. (2013)  Self-Efficacy       Attributions_Success_Effort   
2  Wolters et al. (2013)  Self-Efficacy  Attributions_Success_Environment   
3  Wolters et al. (2013)  Self-Efficacy      Attributions_Failure_Ability   
4  Wolters et al. (2013)  Self-Efficacy       Attributions_Failure_Effort   
5  Wolters et al. (2013)  Self-Efficacy  Attributions_Failure_Environment   
6  Wolters et al. (2013)  Self-Efficacy         Attributions_Failure_Task   

  Feedback_Type Attribution_Type    Setting     r    N   p_value  
0          None         Adaptive  Classroom  0.20  224   p < .01  
1          None         Adaptive  Classroom -0.72  224  p < .001  
2          None      Maladaptive  Classroom -0.16  224   p < .05  
3          None      Maladaptive  Classroom -0.33  224  p < .001  
4          None         Adaptive  Classroom -0.0

### CS 5: Predicting mathematics achievement: The role of perceived feedback, teacher support and self-beliefs (Yildirim & Yildirim, 2017)

In [13]:
# Extracted from Table 1 on page 76
cs5 = pd.DataFrame({
    'Study': ['Yildirim & Yildirim (2017)'] * 2,
    'Variable_A': ['Feedback_Feedback', 'Feedback_Teacher_Support'], 
    'Variable_B': ['Self-Efficacy'] * 2,
    'Feedback_Type': ['Formative'] * 2,
    'Attribution_Type': [None] * 2,
    'Setting': ['Classroom'] * 2,
    'r': [0.10, 0.11],
    'N': [4848] * 2,
    'p_value': ['p < .01'] * 2
})

In [14]:
print(cs5)

                        Study                Variable_A     Variable_B  \
0  Yildirim & Yildirim (2017)         Feedback_Feedback  Self-Efficacy   
1  Yildirim & Yildirim (2017)  Feedback_Teacher_Support  Self-Efficacy   

  Feedback_Type Attribution_Type    Setting     r     N  p_value  
0     Formative             None  Classroom  0.10  4848  p < .01  
1     Formative             None  Classroom  0.11  4848  p < .01  


### CS 6: Perceived formative assessment and student motivational beliefs and self-regulation strategies: a multilevel analysis (Luo & Lim, 2024)

In [15]:
# Extracted from Table 1 on page 292
cs6 = pd.DataFrame({
    'Study': ['Luo & Lim (2024)'],
    'Variable_A': ['Feedback_Formative_Assessment'], 
    'Variable_B': ['Self-Efficacy'],
    'Feedback_Type': ['Formative'],
    'Attribution_Type': [None],
    'Setting': ['Classroom'],
    'r': [0.33],
    'N': [1939],
    'p_value': ['p < .01']
})

In [16]:
print(cs6)

              Study                     Variable_A     Variable_B  \
0  Luo & Lim (2024)  Feedback_Formative_Assessment  Self-Efficacy   

  Feedback_Type Attribution_Type    Setting     r     N  p_value  
0     Formative             None  Classroom  0.33  1939  p < .01  


### CS 7: Teacher Support, Motivation, Learning Strategy Use, and Achievement: A Multilevel Mediation Model (Yildirim, 2012)

In [17]:
# Extracted from Table 1 on page 160
cs7 = pd.DataFrame({
    'Study': ['Yildirim (2012)'],
    'Variable_A': ['Feedback_Teacher_Support'], 
    'Variable_B': ['Self-Efficacy'],
    'Feedback_Type': ['Formative'],
    'Attribution_Type': [None],
    'Setting': ['Classroom'],
    'r': [0.17],
    'N': [4855],
    'p_value': ['p < .01']
})

In [18]:
print(cs7)

             Study                Variable_A     Variable_B Feedback_Type  \
0  Yildirim (2012)  Feedback_Teacher_Support  Self-Efficacy     Formative   

  Attribution_Type    Setting     r     N  p_value  
0             None  Classroom  0.17  4855  p < .01  


### CS 8: Supporting primary students’ mathematical reasoning practice: the effects of formative feedback and the mediating role of self-efficacy (Smit et al., 2023)

In [19]:
# Extracted from Table 1 on page 288
cs8 = pd.DataFrame({
    'Study': ['Smit et al. (2023)'],
    'Variable_A': ['Feedback'], 
    'Variable_B': ['Self-Efficacy'],
    'Feedback_Type': ['Formative'],
    'Attribution_Type': [None],
    'Setting': ['Classroom'],
    'r': [0.21],
    'N': [1261],
    'p_value': ['p < .01']
})

In [20]:
print(cs8)

                Study Variable_A     Variable_B Feedback_Type  \
0  Smit et al. (2023)   Feedback  Self-Efficacy     Formative   

  Attribution_Type    Setting     r     N  p_value  
0             None  Classroom  0.21  1261  p < .01  


In [21]:
# Create dictionary with data from all correlational studies
correlational_studies = {'CS1': cs1, 'CS2': cs2, 'CS3': cs3, 'CS4': cs4, 'CS5': cs5, 'CS6': cs6, 'CS7': cs7, 'CS8': cs8}

In [22]:
# Iterate through dictionary and transform all correlations r to Fisher's z
corr_fisher_z = (
    pd.concat(
        [compute_r_to_fisher_z(df) for label, df in correlational_studies.items()],
        ignore_index=True,   # optional but recommended
    )
    .reset_index(drop=True)
)

In [23]:
print(corr_fisher_z)

                         Study     Study_Type                     Variable_A  \
0                  Phan (2012)  Correlational     Feedback_Verbal_Persuation   
1                  Dong (2023)  Correlational  Attributions_Failure_Internal   
2         van der Kleij (2019)  Correlational                  Self-Efficacy   
3        Wolters et al. (2013)  Correlational                  Self-Efficacy   
4        Wolters et al. (2013)  Correlational                  Self-Efficacy   
5        Wolters et al. (2013)  Correlational                  Self-Efficacy   
6        Wolters et al. (2013)  Correlational                  Self-Efficacy   
7        Wolters et al. (2013)  Correlational                  Self-Efficacy   
8        Wolters et al. (2013)  Correlational                  Self-Efficacy   
9        Wolters et al. (2013)  Correlational                  Self-Efficacy   
10  Yildirim & Yildirim (2017)  Correlational              Feedback_Feedback   
11  Yildirim & Yildirim (2017)  Correlat

## 3. Experimental Studies

For experimental studies, group-level summary statistics were extracted to compute standardized effects. Depending on study design, this included posttest means and standard deviations for treatment and control groups, or pretest and posttest means and standard deviations for single-group (pre–post) designs. Sample sizes were extracted for all groups and conditions.

Experimental designs were coded a priori as either **between-groups** (treatment versus control) or **within-groups** (pre–post without a control group), and effect-size computation was tailored accordingly.

---

### Between-Groups Designs

For experimental studies with independent treatment and control groups, standardized mean differences were computed using group-level posttest outcomes or, where applicable, posttest change scores. The standardized mean difference was calculated as:

$$
d = \frac{\bar{X}_T - \bar{X}_C}{S_p},
$$

where $\bar{X}_T$ and $\bar{X}_C$ denote the treatment and control group means, respectively, and $S_p$ is the pooled standard deviation:

$$
S_p = \sqrt{\frac{(n_T - 1)S_T^2 + (n_C - 1)S_C^2}{n_T + n_C - 2}}.
$$

Here, $n_T$ and $n_C$ denote the sample sizes, and $S_T$ and $S_C$ the corresponding standard deviations for the treatment and control groups. When change scores were reported, the same formula was applied with means and standard deviations referring to the change scores rather than posttest levels.

---

### Within-Groups (Pre–Post) Designs

For experimental studies employing a single-group pre–post design, standardized mean changes were computed as the mean pre–post difference divided by the standard deviation of the change score:

$$
d = \frac{\bar{X}_{post} - \bar{X}_{pre}}{S_{\Delta}}.
$$

When the standard deviation of the change score ($S_{\Delta}$) was not directly reported, it was reconstructed from reported pretest and posttest standard deviations using an assumed pre–post correlation $r$:

$$
S_{\Delta} = \sqrt{S_{pre}^2 + S_{post}^2 - 2r S_{pre} S_{post}}.
$$

A value of $r = 0.75$ was assumed, consistent with recommendations and typical values reported in educational and psychological intervention research (e.g., Papadopoulos, 2025). Sensitivity analyses using alternative values of $r$ were conducted to assess the robustness of the resulting effect sizes.

---

### Conversion to Correlation Metrics

To permit synthesis with correlational studies, standardized mean differences from experimental studies were converted to correlation coefficients using established equivalence relationships.

For between-groups designs, standardized mean differences were converted to an equivalent point-biserial correlation:

$$
r = \frac{d}{\sqrt{d^2 + 4}}.
$$

For within-groups (pre–post) designs, the standardized mean change was first converted to a paired-samples $t$ statistic:

$$
t = d \sqrt{n},
$$

and subsequently transformed to a correlation coefficient:

$$
r = \frac{t}{\sqrt{t^2 + (n - 1)}}.
$$

In both cases, the resulting value represents an **effect-size correlation**, quantifying the strength of association between experimental condition (or time, in pre–post designs) and outcome.

---

### Inference and Meta-Analytic Weighting

Experimental effect-size correlations were transformed to Fisher’s $z$ using:

$$
z = \operatorname{atanh}(r),
$$

with sampling variance:

$$
v_z = \frac{1}{n - 3}.
$$

All experimental effects were meta-analyzed on the Fisher’s $z$ scale using inverse-variance weighting and random-effects models, ensuring a common inferential framework across experimental and correlational designs. Pooled estimates were back-transformed to the correlation scale for interpretation.

In [24]:
# Define the base directory path
data_path = Path(r"C:\Users\rodan\OneDrive - Høgskulen på Vestlandet\Dokumenter\PhD\RQ1\Literature\Final")

### ES 1: Self-Efficacy as a Function of Attributional Feedback (Jain et al., 2007)

In [25]:
# Define full file path by joining directory and filename
file_path = data_path / "ES1_Self-efficacy as a function of attributional feedback.pdf"

# Use camelot to read tables from this file path
es1_tables = camelot.read_pdf(str(file_path), pages='10', flavor='hybrid')

# Extract table 1 on page 10
es1 = (
    es1_tables[0].df
    .iloc[2:]  # drop title/number rows
    .loc[lambda df: pd.to_numeric(df[2], errors='coerce').notnull()]
    .reset_index(drop=True)
)

# Rename columns and convert numeric
es1.columns = ['Condition', 'Gender', 'Pretest_Mean', 'Posttest_Mean', 'Mean_Diff', 'SD', 't_stat']
es1[['Pretest_Mean', 'Posttest_Mean', 'Mean_Diff', 'SD']] = es1[['Pretest_Mean', 'Posttest_Mean', 'Mean_Diff', 'SD']].apply(pd.to_numeric, errors='coerce')

# Clean 'Condition' names, forward-fill, preserve t_stat
es1['t_stat'] = es1['t_stat'].astype(str)
es1['Condition'] = es1['Condition'].replace({'No': 'Control', 'A + E': 'Ability_Effort', 'ABILITY': 'Ability', 'EFFORT': 'Effort', '': pd.NA}).ffill()
es1['N'] = 24

# Combine boys and girls using groupby.apply with include_groups=False
es1_ready = (
    es1.groupby('Condition', group_keys=False)
       .apply(
           lambda g: (
               lambda mean_diff: pd.Series({
                   'Study': 'Jain et al., 2007',
                   'Predictor': 'Feedback_Attributional',
                   'Outcome': 'Self-Efficacy',
                   'Feedback_Type': 'Attributional',
                   'Attribution_Type': None,
                   'Setting': 'Individual',
                   'Condition': g.name,
                   'Design': 'between_groups',
                   'Score_type': 'change_scores',
                   'Mean_diff': mean_diff,
                   'SD': np.sqrt(
                       (((g['N'] - 1) * g['SD']**2).sum() + 
                        (g['N'] * ((g['Posttest_Mean'] - g['Pretest_Mean']) - mean_diff)**2).sum()
                       ) / (g['N'].sum() - 1)
                   ),
                   'N': int(g['N'].sum())
               })
           )(np.average((g["Posttest_Mean"] - g["Pretest_Mean"]), weights=g["N"]))
       , include_groups=False
       )
       .reset_index(drop=True)
)

In [26]:
print(es1)

        Condition Gender  Pretest_Mean  Posttest_Mean  Mean_Diff     SD  \
0          Effort   Boys         56.42          81.09      24.67  11.35   
1          Effort  Girls         48.56          74.17      25.17   7.90   
2         Ability   Boys         50.86          72.87      22.01  14.45   
3         Ability  Girls         52.50          66.93      14.43  15.24   
4  Ability_Effort   Boys         54.55          68.42      13.87  12.71   
5  Ability_Effort  Girls         40.23          55.90      15.67   9.82   
6         Control   Boys         47.83          45.87      -1.96  14.06   
7         Control  Girls         50.95          46.84      -4.00  11.57   

     t_stat   N  
0     7.42*  24  
1    11.18*  24  
2     5.25*  24  
3    2.59**  24  
4     3.74*  24  
5     5.50*  24  
6  0.48(ns)  24  
7  0.99(ns)  24  


In [27]:
print(es1_ready)

               Study               Predictor        Outcome  Feedback_Type  \
0  Jain et al., 2007  Feedback_Attributional  Self-Efficacy  Attributional   
1  Jain et al., 2007  Feedback_Attributional  Self-Efficacy  Attributional   
2  Jain et al., 2007  Feedback_Attributional  Self-Efficacy  Attributional   
3  Jain et al., 2007  Feedback_Attributional  Self-Efficacy  Attributional   

  Attribution_Type     Setting       Condition          Design     Score_type  \
0             None  Individual         Ability  between_groups  change_scores   
1             None  Individual  Ability_Effort  between_groups  change_scores   
2             None  Individual         Control  between_groups  change_scores   
3             None  Individual          Effort  between_groups  change_scores   

   Mean_diff         SD   N  
0     18.220  15.182478  48  
1     14.770  11.272580  48  
2     -3.035  12.783871  48  
3     25.140   9.685425  48  


### ES 2: Sequential attributional feedback and children's achievement behaviours (Schunk, 1984a, 1984b)

In [28]:
# Define full file path by joining directory and filename
file_path = data_path / "ES2_Sequential attributional feedback and children's achievement behaviors.pdf"

# Use camelot to read tables from this file path
es2_tables = camelot.read_pdf(str(file_path), pages='5,7', flavor='hybrid')

# Extract Table 1 on page 1163
es2a = (
    es2_tables[0].df
    .drop([0, 1, 2, 3, 13])  # Drop header junk rows
    .rename(columns={
        0: "Measure",
        1: "Ability_Ability.Mean", 2: "Ability_Ability.SD",
        3: "Ability_Effort.Mean", 4: "Ability_Effort.SD",
        5: "Effort_Ability.Mean", 6: "Effort_Ability.SD",
        7: "Effort_Effort.Mean", 8: "Effort_Effort.SD"
    })
    .T
    .reset_index()
)

# Set final column names
es2a.columns = [
    'Measure', 'SE_pretest', 'SE_posttest', 
    'Skill_pretest', 'Skill_posttest', 
    'Ability', 'Effort', 'Task_ease',
    'Luck', 'Training_progress'
]

# Drop the first row (original header row) and split Measure into Condition & Measure
es2a = (
    es2a.drop(0)
         .assign(**es2a['Measure'].str.split('.', n=1, expand=True).rename(columns={0:'Condition', 1:'Measure'}))
         .loc[:, ['Condition'] + [c for c in es2a.columns if c != 'Condition']]
)

# Fix number incorrectly read as 2,1 instead of 2.1 in row 6, column 'Skill_pretest'
es2a.loc [6, 'Skill_pretest'] = 2.1

# Convert columns to numeric, replacing commas and errors
for col in ['SE_pretest', 'SE_posttest']:
    es2a[col] = pd.to_numeric(es2a[col].str.replace(',', '.'), errors='coerce')

# Assumed pre-post correlation based on typical values reported in educational research (e.g., Dunlosky & Rawson, 2015)
r = 0.75

# Format dataframe for effect size calculations
es2a_ready = (
    es2a.pivot(index='Condition', columns='Measure', values=['SE_pretest', 'SE_posttest'])
        .pipe(lambda d: pd.DataFrame({
            'Study': 'Schunk (1984a)',
            'Predictor': 'Feedback_Attributional',
            'Outcome': 'Self-Efficacy',
            'Feedback_Type': 'Attributional',
            'Attribution_Type': None,
            'Setting': 'Individual',
            'Condition': d.index,
            'Design': 'within_groups',
            'Score_type': 'change_scores',
            'Mean_diff': d['SE_posttest']['Mean'] - d['SE_pretest']['Mean'],
            'SD': np.sqrt(d['SE_pretest']['SD']**2 + d['SE_posttest']['SD']**2 - 2 * r * d['SE_pretest']['SD'] * d['SE_posttest']['SD']),
            'N': 24
        }))
        .reset_index(drop=True)
)

In [29]:
print(es2a)

         Condition Measure  SE_pretest  SE_posttest Skill_pretest  \
1  Ability_Ability    Mean        36.3         87.6           4.1   
2  Ability_Ability      SD        10.2         11.3           1.4   
3   Ability_Effort    Mean        35.4         87.8           3.8   
4   Ability_Effort      SD        10.3          8.3           1.8   
5   Effort_Ability    Mean        32.9         65.3           3.7   
6   Effort_Ability      SD        11.2         16.5           2.1   
7    Effort_Effort    Mean        35.8         72.4           3.5   
8    Effort_Effort      SD        12.1         11.3           2.3   

  Skill_posttest Ability Effort Task_ease  Luck Training_progress  
1           19.3    83.0   82.0      68.0  30.0             207.8  
2            4.2    19.5   16.7      19.3  27.9              45.0  
3           17.4    85.0   95.0      68.0  38.0             186.4  
4            5.9    12.7    9.7      23.5  27.0              50.0  
5           12.3    59.0   89.0      6

In [30]:
print(es2a_ready)

            Study               Predictor        Outcome  Feedback_Type  \
0  Schunk (1984a)  Feedback_Attributional  Self-Efficacy  Attributional   
1  Schunk (1984a)  Feedback_Attributional  Self-Efficacy  Attributional   
2  Schunk (1984a)  Feedback_Attributional  Self-Efficacy  Attributional   
3  Schunk (1984a)  Feedback_Attributional  Self-Efficacy  Attributional   

  Attribution_Type     Setting        Condition         Design     Score_type  \
0             None  Individual  Ability_Ability  within_groups  change_scores   
1             None  Individual   Ability_Effort  within_groups  change_scores   
2             None  Individual   Effort_Ability  within_groups  change_scores   
3             None  Individual    Effort_Effort  within_groups  change_scores   

   Mean_diff         SD   N  
0       51.3   7.670724  24  
1       52.4   6.837032  24  
2       32.4  10.976794  24  
3       36.6   8.306925  24  


In [31]:
# Extract Table 2 on page 1165
es2b = es2_tables[2].df

# Drop the first two rows (0 and 1) since they’re header junk
es2b = es2b.drop([0, 1, 2, 3, 6, 9, 12, 18, 23])

# Reset column headers manually
es2b.columns = [
    "Measure",
    "Ability_Ability.Mean", "Ability_Ability.SD",
    "Ability_Effort.Mean",  "Ability_Effort.SD",
    "Effort_Ability.Mean",  "Effort_Ability.SD",
    "Effort_Effort.Mean",   "Effort_Effort.SD"
]

# Transport dataframe and remane columns
es2b = es2b.T.reset_index()

es2b.columns = [
    'Measure', 'Self-Efficacy_pretest', 'Self-Efficacy_posttest', 
    'Skill_pretest', 'Skill_posttest', 
    'Attributions_Success_Ability_pretest',  'Attributions_Success_Ability_posttest', 
    'Attributions_Success_Effort_pretest', 'Attributions_Success_Effort_posttest', 
    'Attributions_Success_Task_Ease_pretest', 'Attributions_Success_Task_Ease_posttest',
    'Attributions_Success_Luck_pretest', 'Attributions_Success_Luck_posttest', 
    'Training_progress', 'Training_perception'
]

# Drop old column names in row 0
es2b.drop(0, inplace=True)

# Split measure column into condition and measure columns
es2b[['Condition', 'Measure']] = es2b['Measure'].str.split('.', n=1, expand=True)

# Create a list of new column names
es2b_cols = ["Condition"] + [col for col in es2b.columns if col != "Condition"]

# Assign new column names
es2b = es2b[es2b_cols]

# Manually update values incorectly read by Camelot
es2b.loc[2, 'Attributions_Success_Task_Ease_pretest'] = 12.9
es2b.loc[8, 'Training_progress'] = 45.4

# Convert columns to numeric, replacing commas and errors
for col in es2b.columns[2:]:
    es2b[col] = pd.to_numeric(es2b[col], errors='coerce')

# Assumed pre-post correlation based on typical values reported in educational research (e.g., Dunlosky & Rawson, 2015)
r = 0.75

# Create a list of outcomes to iterate through
outcomes = ['Self-Efficacy', 'Skill', 'Attributions_Success_Ability', 'Attributions_Success_Effort', 
            'Attributions_Success_Task_Ease', 'Attributions_Success_Luck']

attribution_type = [None, None, 'Adaptive', 'Adaptive', 'Maladaptive', 'Maladaptive']

# Pivot dataframe by condition
d = es2b.pivot(index='Condition', columns='Measure')

# Format dataframe for effect size calculations
es2b_ready = pd.concat([
    pd.DataFrame({
        'Study': 'Schunk (1984b)',
        'Predictor': 'Feedback_Attributional',
        'Outcome': outcome,
        'Feedback_Type': 'Attributional',
        'Attribution_Type': attribution,
        'Setting': 'Individual',
        'Condition': d.index,
        'Design': 'within_groups',
        'Score_type': 'change_scores',
        'Mean_diff': d[(f'{outcome}_posttest', 'Mean')] - d[(f'{outcome}_pretest', 'Mean')],
        'SD': np.sqrt(
            d[(f'{outcome}_pretest', 'SD')]**2 +
            d[(f'{outcome}_posttest', 'SD')]**2 -
            2 * r * d[(f'{outcome}_pretest', 'SD')] * d[(f'{outcome}_posttest', 'SD')]
        ),
        'N': 10
    }) for outcome, attribution in zip(outcomes, attribution_type)
]).reset_index(drop=True)

In [32]:
print(es2b)

         Condition Measure  Self-Efficacy_pretest  Self-Efficacy_posttest  \
1  Ability_Ability    Mean                   33.5                    86.7   
2  Ability_Ability      SD                    9.6                    15.0   
3   Ability_Effort    Mean                   31.2                    86.6   
4   Ability_Effort      SD                    9.0                    16.4   
5   Effort_Ability    Mean                   35.2                    66.7   
6   Effort_Ability      SD                   17.8                    19.5   
7    Effort_Effort    Mean                   31.9                    64.6   
8    Effort_Effort      SD                    8.1                    12.0   

   Skill_pretest  Skill_posttest  Attributions_Success_Ability_pretest  \
1            3.0            16.6                                  40.0   
2            2.1             3.0                                  19.4   
3            2.8            16.4                                  42.0   
4         

In [33]:
print(es2b_ready)

             Study               Predictor                         Outcome  \
0   Schunk (1984b)  Feedback_Attributional                   Self-Efficacy   
1   Schunk (1984b)  Feedback_Attributional                   Self-Efficacy   
2   Schunk (1984b)  Feedback_Attributional                   Self-Efficacy   
3   Schunk (1984b)  Feedback_Attributional                   Self-Efficacy   
4   Schunk (1984b)  Feedback_Attributional                           Skill   
5   Schunk (1984b)  Feedback_Attributional                           Skill   
6   Schunk (1984b)  Feedback_Attributional                           Skill   
7   Schunk (1984b)  Feedback_Attributional                           Skill   
8   Schunk (1984b)  Feedback_Attributional    Attributions_Success_Ability   
9   Schunk (1984b)  Feedback_Attributional    Attributions_Success_Ability   
10  Schunk (1984b)  Feedback_Attributional    Attributions_Success_Ability   
11  Schunk (1984b)  Feedback_Attributional    Attributions_Succe

### ES 3: Strategy Training and Attributional Feedback With Learning Disabled Students (Schunk & Cox, 1986)

In [34]:
# Extract pre and post test data from Table 1 on page 7
es3 = pd.DataFrame({
    'Condition': [
        'Effort_Performance', 'Effort_Performance',
        'Performance_Effort', 'Performance_Effort',
        'Control', 'Control'
    ],
    'Measure': [
        'Mean', 'SD',
        'Mean', 'SD',
        'Mean', 'SD'
    ],
    'Self-Efficacy_pretest': [55.0, 30.7, 57.7, 25.9, 55.9, 24.0],
    'Self-Efficacy_posttest': [78.7, 18.2, 77.2, 18.4, 64.4, 17.8],
    'Skill_pretest': [9.0, 7.4, 9.1, 7.8, 7.7, 5.8],
    'Skill_posttest': [16.3, 6.3, 15.8, 5.5, 11.7, 4.5],
    'Attributions_Success_Ability_pretest': [54.7, 30.1, 54.0, 32.2, 53.3, 30.7],
    'Attributions_Success_Ability_posttest': [65.3, 29.3, 64.7, 29.1, 59.0, 25.9],
    'Attributions_Success_Effort_pretest': [74.7, 28.0, 73.7, 26.2, 68.0, 28.7],
    'Attributions_Success_Effort_posttest': [87.3, 15.5, 72.7, 23.9, 60.3, 27.7],
    'Attributions_Success_Task_Ease_pretest': [73.7, 24.4, 72.0, 23.4, 66.3, 26.2],
    'Attributions_Success_Task_Ease_posttest': [72.0, 24.0, 68.3, 21.0, 70.0, 24.9],
    'Attributions_Success_Luck_pretest': [55.7, 30.3, 53.3, 27.8, 51.0, 28.2],
    'Attributions_Success_Luck_posttest': [47.3, 28.6, 43.0, 30.1, 59.0, 25.4]
})

# Assumed pre-post correlation based on typical values reported in educational research (e.g., Dunlosky & Rawson, 2015)
r = 0.75

# Create a list of outcomes to iterate through
outcomes = ['Self-Efficacy', 'Skill', 'Attributions_Success_Ability', 'Attributions_Success_Effort', 
            'Attributions_Success_Task_Ease', 'Attributions_Success_Luck']

attribution_type = [None, None, 'Adaptive', 'Adaptive', 'Maladaptive', 'Maladaptive']

# Pivot dataframe by condition 
d = es3.pivot(index='Condition', columns='Measure')

# Format dataframe for effect size calculations
es3_ready = pd.concat([
    pd.DataFrame({
        'Study': 'Schunk & Cox (1986)',
        'Predictor': 'Feedback_Attributional',
        'Outcome': outcome,
        'Feedback_Type': 'Attributional',
        'Attribution_Type': attribution,
        'Setting': 'Individual',
        'Condition': d.index,
        'Design': 'between_groups',
        'Score_type': 'change_scores',
        'Mean_diff': d[(f'{outcome}_posttest', 'Mean')] - d[(f'{outcome}_pretest', 'Mean')],
        'SD': np.sqrt(
            d[(f'{outcome}_pretest', 'SD')]**2 +
            d[(f'{outcome}_posttest', 'SD')]**2 -
            2 * r * d[(f'{outcome}_pretest', 'SD')] * d[(f'{outcome}_posttest', 'SD')]
        ),
        'N': 30
    }) for outcome, attribution in zip(outcomes, attribution_type)
]).reset_index(drop=True)

In [35]:
print(es3)

            Condition Measure  Self-Efficacy_pretest  Self-Efficacy_posttest  \
0  Effort_Performance    Mean                   55.0                    78.7   
1  Effort_Performance      SD                   30.7                    18.2   
2  Performance_Effort    Mean                   57.7                    77.2   
3  Performance_Effort      SD                   25.9                    18.4   
4             Control    Mean                   55.9                    64.4   
5             Control      SD                   24.0                    17.8   

   Skill_pretest  Skill_posttest  Attributions_Success_Ability_pretest  \
0            9.0            16.3                                  54.7   
1            7.4             6.3                                  30.1   
2            9.1            15.8                                  54.0   
3            7.8             5.5                                  32.2   
4            7.7            11.7                                  53.

In [36]:
print(es3_ready)

                  Study               Predictor  \
0   Schunk & Cox (1986)  Feedback_Attributional   
1   Schunk & Cox (1986)  Feedback_Attributional   
2   Schunk & Cox (1986)  Feedback_Attributional   
3   Schunk & Cox (1986)  Feedback_Attributional   
4   Schunk & Cox (1986)  Feedback_Attributional   
5   Schunk & Cox (1986)  Feedback_Attributional   
6   Schunk & Cox (1986)  Feedback_Attributional   
7   Schunk & Cox (1986)  Feedback_Attributional   
8   Schunk & Cox (1986)  Feedback_Attributional   
9   Schunk & Cox (1986)  Feedback_Attributional   
10  Schunk & Cox (1986)  Feedback_Attributional   
11  Schunk & Cox (1986)  Feedback_Attributional   
12  Schunk & Cox (1986)  Feedback_Attributional   
13  Schunk & Cox (1986)  Feedback_Attributional   
14  Schunk & Cox (1986)  Feedback_Attributional   
15  Schunk & Cox (1986)  Feedback_Attributional   
16  Schunk & Cox (1986)  Feedback_Attributional   
17  Schunk & Cox (1986)  Feedback_Attributional   

                           Out

### ES 4: Effects of Internally focused feedback (Craven et al., 1991)

In [37]:
# Define full file path by joining directory and filename
file_path = data_path / "../Search #2/ES4_Effects of Internally focused feedback.pdf"

# Use camelot to read tables from this file path
es4_tables = camelot.read_pdf(str(file_path), pages='8', flavor='hybrid')

# Extract post test means and standard deviations Table 1 on page 23
es4 = es4_tables[0].df.iloc[15:].reset_index(drop=True)

# Set headers and transpose
es4.columns = ["Measure", "Researcher.Mean", "Researcher.SD", 
               "Teacher.Mean", "Teacher.SD", 
               "Teacher_Researcher.Mean", "Teacher_Researcher.SD", 
               "Control.Mean", "Control.SD"]

# Transpose dataframe
es4 = es4.T.reset_index()

# Rename columns in transposed dataframe
es4.columns = [
    val.split('/')[0] + '-' + val.split('/')[1].capitalize()
    if '/' in val else val
    for val in es4.iloc[0]
]

# Drop index of dataframe
es4 = es4.drop(0).reset_index(drop=True)

# Split Condition and Measure
es4[['Condition', 'Measure']] = es4['Measure'].str.split('.', n=1, expand=True)

# Pivot dataframe by condition 
d = es4.pivot(index='Condition', columns='Measure')

# Define your mapping from old level-0 names to new names
mapping = {'Success-Ability': 'Attributions_Success_Ability', 'Success-Effort': 'Attributions_Success_Effort',
           'Success-External': 'Attributions_Success_External', 'Failure-Ability': 'Attributions_Failure_Ability',
           'Failure-Effort': 'Attributions_Failure_Effort', 'Failure-External': 'Attributions_Failure_External'}

# Get current level-0 names
level_0 = d.columns.get_level_values(0)
level_1 = d.columns.get_level_values(1)

# Map them to new names automatically
new_level_0 = [mapping.get(x, x) for x in level_0]

# Rebuild MultiIndex
d.columns = pd.MultiIndex.from_arrays([new_level_0, level_1], names=[None, 'Measure'])

# Create a list of outcomes to iterate through
outcomes = d.columns.get_level_values(0).unique().to_list()
attribution_type = ['Adaptive', 'Adaptive', 'Maladaptive', 'Maladaptive', 'Adaptive', 'Maladaptive']

# Format dataframe for effect size calculations
es4_ready = pd.concat([
    pd.DataFrame({
        'Study': 'Craven et al. (1991)',
        'Predictor': 'Feedback_Attributional',
        'Outcome': outcome,
        'Feedback_Type': 'Attributional',
        'Attribution_Type': attribution,
        'Setting': ['Classroom', 'Small_Groups', 'Classroom', 'Classroom'], 
        'Condition': d.index,
        'Design': 'between_groups',
        'Score_type': 'post',
        'Mean_diff': d[outcome, 'Mean'],
        'SD': d[outcome, 'SD'],
        'N': [79] + [27] * 3
    }) for outcome, attribution in zip(outcomes, attribution_type)
]).reset_index(drop=True)

In [38]:
print(es4)

  Measure Success-Ability Success-Effort Success-External Failure-Ability  \
0    Mean            4.31           4.48             2.76            2.09   
1      SD            0.71           0.52             0.92            1.09   
2    Mean            3.89           4.08             2.84            2.27   
3      SD            0.82           0.80             0.85            1.03   
4    Mean            4.15           4.40             3.19            2.06   
5      SD            0.96           0.60             1.07            1.03   
6    Mean            4.03           4.17             2.86            2.26   
7      SD            0.80           0.72             1.01            0.93   

  Failure-Effort Failure-External           Condition  
0           2.34             2.48          Researcher  
1           1.11             0.90          Researcher  
2           2.58             2.70             Teacher  
3           0.98             0.78             Teacher  
4           2.39          

In [39]:
print(es4_ready)

                   Study               Predictor  \
0   Craven et al. (1991)  Feedback_Attributional   
1   Craven et al. (1991)  Feedback_Attributional   
2   Craven et al. (1991)  Feedback_Attributional   
3   Craven et al. (1991)  Feedback_Attributional   
4   Craven et al. (1991)  Feedback_Attributional   
5   Craven et al. (1991)  Feedback_Attributional   
6   Craven et al. (1991)  Feedback_Attributional   
7   Craven et al. (1991)  Feedback_Attributional   
8   Craven et al. (1991)  Feedback_Attributional   
9   Craven et al. (1991)  Feedback_Attributional   
10  Craven et al. (1991)  Feedback_Attributional   
11  Craven et al. (1991)  Feedback_Attributional   
12  Craven et al. (1991)  Feedback_Attributional   
13  Craven et al. (1991)  Feedback_Attributional   
14  Craven et al. (1991)  Feedback_Attributional   
15  Craven et al. (1991)  Feedback_Attributional   
16  Craven et al. (1991)  Feedback_Attributional   
17  Craven et al. (1991)  Feedback_Attributional   
18  Craven e

### ES 5: Grade expectations the motivational consequences of performance feedback on a summative assessment (Koenka, 2022)

In [40]:
# Extract pre and post test data from Table 5 on page 101
es5 = pd.DataFrame({
    'Study': 'Koenka, 2022',
    'Predictor': 'Feedback_Performance',
    'Outcome': 'Self-Efficacy',
    'Feedback_Type': 'Performance',
    'Attribution_Type': None,
    'Setting': 'Classroom',
    'Condition': ['Grades_Written_Comments', 'Grades', 'Written_Comments', 'Control'],
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    'W2_Mean': [3.88, 4.04, 3.93, 3.75],
    'W2_SD': [0.79, 0.78, 0.92, 0.91],
    'W2_N': [57, 33, 23, 40],
    'W3_Mean': [3.61, 3.98, 3.91, 3.72],
    'W3_SD': [0.89, 0.80, 0.80, 0.80],
    'W3_N': [59, 33, 23, 34]
})

# Assumed pre-post correlation based on typical values reported in educational research (Papadopoulos, 2025)
r = 0.75

# Format dataframe for effect size calculations
es5_ready = es5.pipe(
    lambda d: d.assign(
        Mean_diff = d['W3_Mean'] - d['W2_Mean'],
        SD = np.sqrt(d['W2_SD']**2 + d['W3_SD']**2 - 2 * r * d['W2_SD'] * d['W3_SD']),
        N = d[['W2_N', 'W3_N']].min(axis=1)
    )
).pipe(
    lambda d: d.drop(d.filter(regex=r'W[23]|SD_pooled').columns, axis=1)
)

In [41]:
print(es5)

          Study             Predictor        Outcome Feedback_Type  \
0  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
1  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
2  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
3  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   

  Attribution_Type    Setting                Condition          Design  \
0             None  Classroom  Grades_Written_Comments  between_groups   
1             None  Classroom                   Grades  between_groups   
2             None  Classroom         Written_Comments  between_groups   
3             None  Classroom                  Control  between_groups   

      Score_type  W2_Mean  W2_SD  W2_N  W3_Mean  W3_SD  W3_N  
0  change_scores     3.88   0.79    57     3.61   0.89    59  
1  change_scores     4.04   0.78    33     3.98   0.80    33  
2  change_scores     3.93   0.92    23     3.91   0.80    23  
3  change_scores     3.75 

In [42]:
print(es5_ready)

          Study             Predictor        Outcome Feedback_Type  \
0  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
1  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
2  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   
3  Koenka, 2022  Feedback_Performance  Self-Efficacy   Performance   

  Attribution_Type    Setting                Condition          Design  \
0             None  Classroom  Grades_Written_Comments  between_groups   
1             None  Classroom                   Grades  between_groups   
2             None  Classroom         Written_Comments  between_groups   
3             None  Classroom                  Control  between_groups   

      Score_type  Mean_diff        SD   N  
0  change_scores      -0.27  0.601290  57  
1  change_scores      -0.06  0.558928  33  
2  change_scores      -0.02  0.618385  23  
3  change_scores      -0.03  0.613270  34  


### ES 6: Formative assessment in mathematics Mediated by feedback's perceived usefulness and students’ self-efficacy (Rakoczy et al., 2017)

In [43]:
# Extract pre and post test data from Table 1 on page 162
es6 = pd.DataFrame({
    'Study': 'Rakoczy et al. (2017)',
    'Predictor': 'Feedback_Formative',
    'Outcome': 'Self-Efficacy',
    'Feedback_Type': 'Formative',
    'Attribution_Type': None,
    'Setting': 'Classroom',
    "Condition": ["Written_Process_Feedback", "Control"],
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    "Pre_Mean": [2.35, 2.37],
    "Pre_SD": [0.60, 0.55],
    "Pre_N": [259, 361],  # replace with actual sample sizes
    "Post_Mean": [2.82, 2.68],
    "Post_SD": [0.66, 0.62],
    "Post_N": [259, 361]  # replace with actual sample sizes
})

# Assumed pre-post correlation based on typical values reported in educational research (Papadopoulos, 2025)
r = 0.75

# Format dataframe for effect size calculations
es6_ready = es6.pipe(
    lambda d: d.assign(
        Mean_diff = d['Post_Mean'] - d['Pre_Mean'],
        SD = np.sqrt(d['Pre_SD']**2 + d['Post_SD']**2 - 2 * r * d['Pre_SD'] * d['Post_SD']),
        N = d[['Pre_N', 'Post_N']].min(axis=1)
    )
).pipe(
    lambda d: d.drop(d.filter(regex=r'Pre_|Post_').columns, axis=1)
)

In [44]:
print(es6_ready)

                   Study           Predictor        Outcome Feedback_Type  \
0  Rakoczy et al. (2017)  Feedback_Formative  Self-Efficacy     Formative   
1  Rakoczy et al. (2017)  Feedback_Formative  Self-Efficacy     Formative   

  Attribution_Type    Setting                 Condition          Design  \
0             None  Classroom  Written_Process_Feedback  between_groups   
1             None  Classroom                   Control  between_groups   

      Score_type  Mean_diff        SD    N  
0  change_scores       0.47  0.448999  259  
1  change_scores       0.31  0.418808  361  


### ES 7: Benefits of Integrating an Explicit Self-Efficacy Intervention With Calculation Strategy Training for Low-Performing Elementary Students (Koponen et al., 2021)

In [45]:
# Extract pre and post means differences from Table 4 on page 10
es7a_ready = pd.DataFrame({
    'Study': 'Koponen et al. (2021)',
    'Predictor': 'Feedback_Self-Efficacy',
    'Outcome': 'Self-Efficacy',
    'Feedback_Type': 'Self-Efficacy',
    'Attribution_Type': None,
    'Setting': 'Small_Groups',
    "Condition": ["Self-Efficacy+Skill", "Skill", "Control"],
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    "Mean_diff": [4.86, 2.60, 0.53],
    "SD": [8.98, 4.56, 5.13],
    "N": [28, 32, 51]
})

# Extract pre and post means differences for students with low intial self-efficacy from Table 4 on page 10
es7b_ready = pd.DataFrame({
    'Study': 'Koponen et al. (2021)',
    'Predictor': 'Feedback_Self-Efficacy',
    'Outcome': 'Self-Efficacy',
    'Feedback_Type': 'Self-Efficacy',
    'Attribution_Type': None,
    'Setting': 'Small_Groups',
    "Condition": ["Self-Efficacy+Skill_low", "Skill_low", "Control"],
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    "Mean_diff": [9.23, 2.58, 2.50],
    "SD": [9.82, 4.75, 6.02],
    "N": [13, 17, 12]
})

In [46]:
print(es7a_ready)

                   Study               Predictor        Outcome  \
0  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   
1  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   
2  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   

   Feedback_Type Attribution_Type       Setting            Condition  \
0  Self-Efficacy             None  Small_Groups  Self-Efficacy+Skill   
1  Self-Efficacy             None  Small_Groups                Skill   
2  Self-Efficacy             None  Small_Groups              Control   

           Design     Score_type  Mean_diff    SD   N  
0  between_groups  change_scores       4.86  8.98  28  
1  between_groups  change_scores       2.60  4.56  32  
2  between_groups  change_scores       0.53  5.13  51  


In [47]:
print(es7b_ready)

                   Study               Predictor        Outcome  \
0  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   
1  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   
2  Koponen et al. (2021)  Feedback_Self-Efficacy  Self-Efficacy   

   Feedback_Type Attribution_Type       Setting                Condition  \
0  Self-Efficacy             None  Small_Groups  Self-Efficacy+Skill_low   
1  Self-Efficacy             None  Small_Groups                Skill_low   
2  Self-Efficacy             None  Small_Groups                  Control   

           Design     Score_type  Mean_diff    SD   N  
0  between_groups  change_scores       9.23  9.82  13  
1  between_groups  change_scores       2.58  4.75  17  
2  between_groups  change_scores       2.50  6.02  12  


### ES 8: Effects of Effort Attributional Feedback on Children's Perceived Self-Efficacy and Achievement (Schunk, 1982)

In [48]:
# Extract pre and post test data from Table 1 on page 4
es8 = pd.DataFrame({
    'Study': 'Schunk (1982)',
    'Predictor': 'Feedback_Attributional',
    'Outcome': ['Skill'] * 4 + ['Persistence_Easy_Tasks'] * 4 + ['Persistence_Hard_Tasks'] * 4 + ['Self-Efficacy'] * 4,
    'Feedback_Type': 'Attributional',
    'Attribution_Type': None,
    'Setting': 'Individual',
    'Condition': ['Attributions_Past+Monitoring', 'Attributions_Future+Monitoring', 'Monitoring', 'Control'] * 4,
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    'Pretest_Mean': [1.9, 2.0, 1.5, 1.2,
                    23.2, 25.8, 27.0, 33.1,
                    22.2, 23.2, 23.1, 25.6,
                    44.3, 47.4, 48.8, 49.8],
    'Pretest_SD':  [1.9, 2.4, 1.3, 1.8,
                   12.4, 11.9, 13.9, 16.6,
                   10.8, 13.1, 15.6, 17.9,
                   13.2, 12.6, 12.6, 12.8],
    'Posttest_Mean': [17.3, 6.4, 5.1, 2.7,
                      20.2, 23.7, 25.4, 20.8,
                      30.6, 24.0, 25.1, 20.5,
                      82.3, 52.8, 60.5, 53.4],
    'Posttest_SD': [5.0, 7.4, 6.2, 2.4,
                   7.6, 10.6, 10.2, 8.0,
                   11.1, 11.8, 14.6, 13.2,
                   11.2, 10.4, 16.4, 12.2],
})

# Format dataframe for effect size calculations
es8_ready = es8.pipe(
    lambda d: d.assign(
        Mean_diff = d['Posttest_Mean'] - d['Pretest_Mean'],
        SD = np.sqrt(d['Pretest_SD']**2 + d['Posttest_SD']**2 - 2 * r * d['Pretest_SD'] * d['Posttest_SD']),
        N = 10
    )
).pipe(
    lambda d: d.drop(d.filter(regex=r'Pretest|Posttest').columns, axis=1)
)

In [49]:
print(es8_ready)

            Study               Predictor                 Outcome  \
0   Schunk (1982)  Feedback_Attributional                   Skill   
1   Schunk (1982)  Feedback_Attributional                   Skill   
2   Schunk (1982)  Feedback_Attributional                   Skill   
3   Schunk (1982)  Feedback_Attributional                   Skill   
4   Schunk (1982)  Feedback_Attributional  Persistence_Easy_Tasks   
5   Schunk (1982)  Feedback_Attributional  Persistence_Easy_Tasks   
6   Schunk (1982)  Feedback_Attributional  Persistence_Easy_Tasks   
7   Schunk (1982)  Feedback_Attributional  Persistence_Easy_Tasks   
8   Schunk (1982)  Feedback_Attributional  Persistence_Hard_Tasks   
9   Schunk (1982)  Feedback_Attributional  Persistence_Hard_Tasks   
10  Schunk (1982)  Feedback_Attributional  Persistence_Hard_Tasks   
11  Schunk (1982)  Feedback_Attributional  Persistence_Hard_Tasks   
12  Schunk (1982)  Feedback_Attributional           Self-Efficacy   
13  Schunk (1982)  Feedback_Attrib

### ES 9: Ability Versus Effort Attributional Feedback Differential Effects on Self-Efficacy and Achievement (Schunk, 1983)

In [50]:
# Extract pre and post test data from Table 1 on page 6
es9 = pd.DataFrame({
    'Study': 'Schunk (1983)',
    'Predictor': 'Feedback_Attributional',
    'Outcome': ['Skill'] * 4 + ['Persistence'] * 4 + ['Self-Efficacy'] * 4,
    'Feedback_Type': 'Attributional',
    'Attribution_Type': None,
    'Setting': 'Individual',
    'Condition': ['Ability', 'Effort', 'Ability_Effort', 'Control'] * 3,
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    'Pretest_Mean': [4.5, 4.5, 4.6, 4.3,
                    47.5, 48.1, 53.2, 46.8,
                    39.6, 37.1, 35.8, 36.5],
    'Pretest_SD':  [5.5, 4.9, 4.0, 5.6,
                   18.3, 20.1, 15.8, 20.7,
                   15.9, 13.3, 17.1, 13.4],
    'Posttest_Mean': [18.8, 13.2, 12.6, 8.0,
                     43.0, 43.0, 45.0, 43.1,
                     80.9, 60.4, 60.0, 43.3],
    'Posttest_SD': [3.7, 3.3, 6.3, 4.7,
                   18.3, 22.8, 15.8, 8.8,
                   13.8, 17.4, 26.6, 9.8],
})

# Format dataframe for effect size calculations
es9_ready = es9.pipe(
    lambda d: d.assign(
        Mean_diff = d['Posttest_Mean'] - d['Pretest_Mean'],
        SD = np.sqrt(d['Pretest_SD']**2 + d['Posttest_SD']**2 - 2 * r * d['Pretest_SD'] * d['Posttest_SD']),
        N = 11
    )
).pipe(
    lambda d: d.drop(d.filter(regex=r'Pretest|Posttest').columns, axis=1)
)

In [51]:
print(es9_ready)

            Study               Predictor        Outcome  Feedback_Type  \
0   Schunk (1983)  Feedback_Attributional          Skill  Attributional   
1   Schunk (1983)  Feedback_Attributional          Skill  Attributional   
2   Schunk (1983)  Feedback_Attributional          Skill  Attributional   
3   Schunk (1983)  Feedback_Attributional          Skill  Attributional   
4   Schunk (1983)  Feedback_Attributional    Persistence  Attributional   
5   Schunk (1983)  Feedback_Attributional    Persistence  Attributional   
6   Schunk (1983)  Feedback_Attributional    Persistence  Attributional   
7   Schunk (1983)  Feedback_Attributional    Persistence  Attributional   
8   Schunk (1983)  Feedback_Attributional  Self-Efficacy  Attributional   
9   Schunk (1983)  Feedback_Attributional  Self-Efficacy  Attributional   
10  Schunk (1983)  Feedback_Attributional  Self-Efficacy  Attributional   
11  Schunk (1983)  Feedback_Attributional  Self-Efficacy  Attributional   

   Attribution_Type     

### ES 10: Modeling and Attributional Effects on Children's Achievement A Self-Efficacy Analysis (Schunk, 1981)

In [52]:
# Extract pre and post test data from Table 1 on page 6
es10 = pd.DataFrame({
    'Study': 'Schunk (1981)',
    'Predictor': 'Feedback_Attributional',
    'Outcome': ['Skill'] * 5 + ['Persistence'] * 5 + ['Self-Efficacy'] * 5,
    'Feedback_Type': 'Attributional',
    'Attribution_Type': None,
    'Setting': 'Individual',
    'Condition': ['Cognitive_Modeling_Effort', 'Cognitive_Modeling',
                  'Didactic_Effort', 'Didactic', 'Control'] * 3,
    'Design': 'between_groups',
    'Score_type': 'change_scores',
    'Pretest_Mean': [2.2, 2.7, 2.4, 2.3, 1.6,
                    36.0, 53.3, 45.9, 65.0, 28.9,
                    2.0, 2.3, 2.2, 3.2, 1.8,],
    'Pretest_SD': [1.6, 2.5, 1.7, 1.4, 1.6,
                  35.6, 53.3, 34.1, 61.8, 23.4,
                  1.7, 2.1, 1.9, 2.7, 2.4],
    'Posttest_Mean': [8.3, 7.8, 6.3, 7.0, 1.8,
                     99.7, 79.5, 100.1, 97.0, 13.6,
                     7.5, 7.2, 7.4, 6.3, 2.8],
    'Posttest_SD': [3.6, 3.0, 2.7, 3.1, 2.1,
                   53.3, 41.6, 57.8, 56.6, 10.6,
                   4.7, 5.0, 4.2, 4.4, 4.1]
})

# Format dataframe for effect size calculations
es10_ready = es10.pipe(
    lambda d: d.assign(
        Mean_diff = d['Posttest_Mean'] - d['Pretest_Mean'],
        SD = np.sqrt(d['Pretest_SD']**2 + d['Posttest_SD']**2 - 2 * r * d['Pretest_SD'] * d['Posttest_SD']),
        N = ([12] * 4 + [8]) * 3
    )
).pipe(
    lambda d: d.drop(d.filter(regex=r'Pretest|Posttest').columns, axis=1)
)

In [53]:
print(es10_ready)

            Study               Predictor        Outcome  Feedback_Type  \
0   Schunk (1981)  Feedback_Attributional          Skill  Attributional   
1   Schunk (1981)  Feedback_Attributional          Skill  Attributional   
2   Schunk (1981)  Feedback_Attributional          Skill  Attributional   
3   Schunk (1981)  Feedback_Attributional          Skill  Attributional   
4   Schunk (1981)  Feedback_Attributional          Skill  Attributional   
5   Schunk (1981)  Feedback_Attributional    Persistence  Attributional   
6   Schunk (1981)  Feedback_Attributional    Persistence  Attributional   
7   Schunk (1981)  Feedback_Attributional    Persistence  Attributional   
8   Schunk (1981)  Feedback_Attributional    Persistence  Attributional   
9   Schunk (1981)  Feedback_Attributional    Persistence  Attributional   
10  Schunk (1981)  Feedback_Attributional  Self-Efficacy  Attributional   
11  Schunk (1981)  Feedback_Attributional  Self-Efficacy  Attributional   
12  Schunk (1981)  Feedba

In [54]:
# Create a dictionary of study names and mean differences from the study to iterate through
experimental_studies = {'ES1': es1_ready, 'ES2a': es2a_ready, 'ES2b': es2b_ready, 'ES3': es3_ready, 'ES4': es4_ready, 'ES5': es5_ready, 
                        'ES6': es6_ready, 'ES7a': es7a_ready, 'ES7b': es7b_ready, 'ES8': es8_ready, 'ES9': es9_ready, 'ES10': es10_ready}

In [55]:
# Calculate effect sizes in experimental studies
exp_fisher_z = pd.concat([compute_cohen_d_to_fisher_z(df) for label, df in experimental_studies.items()]).reset_index(drop=True)

In [56]:
print(exp_fisher_z.head())

               Study    Study_Type               Predictor        Outcome  \
0  Jain et al., 2007  Experimental  Feedback_Attributional  Self-Efficacy   
1  Jain et al., 2007  Experimental  Feedback_Attributional  Self-Efficacy   
2  Jain et al., 2007  Experimental  Feedback_Attributional  Self-Efficacy   
3     Schunk (1984a)  Experimental  Feedback_Attributional  Self-Efficacy   
4     Schunk (1984a)  Experimental  Feedback_Attributional  Self-Efficacy   

   Feedback_Type Attribution_Type     Setting        Condition  \
0  Attributional             None  Individual          Ability   
1  Attributional             None  Individual   Ability_Effort   
2  Attributional             None  Individual           Effort   
3  Attributional             None  Individual  Ability_Ability   
4  Attributional             None  Individual   Ability_Effort   

           Design     Score_type  ...  Hedges_g  r_effect  Fisher_z      SE_z  \
0  between_groups  change_scores  ...  1.502365  0.603688  

In [57]:
# Save fisher z results as CSV file
corr_fisher_z.to_csv('corr_fisher_z.csv', index=False, encoding="utf-8")
exp_fisher_z.to_csv('exp_fisher_z.csv', index=False, encoding="utf-8")